In [63]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

BASE_DIR = Path.cwd().parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "events_with_car.csv", sep=";")

print("Loaded:", df.shape)
df.head()

Loaded: (72, 18)


,event_id,event_date,trading_date,ticker,publisher,studio,is_rockstar,game,franchise,event_type,sentiment,impact_expectation_manual,adj_close,return,market_return,AR_event,CAR_m1_p1,CAR_m5_p5
0,ATVI_2019_CODMOBILE_LAUNCH,2019-10-01,2019-10-01,ATVI,Activision,TiMi Studios,0,Call of Duty: Mobile,Call of Duty,Release,Positive,Medium,94.157463,-0.010938,-0.012258,-0.001198,0.005278,-0.013979
1,ATVI_2019_CODMW_RELEASE,2019-10-25,2019-10-25,ATVI,Activision,Infinity Ward,0,Call of Duty: Modern Warfare,Call of Duty,Release,Positive,High,93.729248,0.003438,0.004073,-0.000318,0.000526,-0.002328
2,ATVI_2020_WARCRAFT3_REFORGED,2020-01-28,2020-01-28,ATVI,Activision Blizzard,Blizzard,0,Warcraft III: Reforged,Warcraft,Controversy,Negative,Medium,108.901489,0.012120,0.010054,0.003422,0.003371,-0.024705
3,ATVI_2020_WARZONE_LAUNCH,2020-03-10,2020-03-10,ATVI,Activision,Infinity Ward,0,Call of Duty: Warzone,Call of Duty,Release,Positive,High,100.609787,0.024274,0.049396,-0.016937,0.001569,-0.031189
4,ATVI_2021_LAWSUIT,2021-07-20,2021-07-20,ATVI,Activision Blizzard,NaN,0,NaN,Activision,Controversy,negative,high,137.807159,-0.001204,0.015163,-0.014124,-0.024080,0.003237


In [64]:
# Normalize categorical columns
for col in ["publisher", "event_type", "sentiment", "franchise"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", "")
    )

In [65]:
# GTA dummy
df["franchise_gta"] = (df["franchise"] == "gta").astype(int)

# Force numeric types
num_cols = ["CAR_m1_p1", "AR_event", "market_return", "is_rockstar", "franchise_gta"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where CAR_m1_p1 is missing (no dependent variable)
df = df.dropna(subset=["CAR_m1_p1"])

final_df = df

print("After dropping Na CAR rows:", df.shape)

After dropping Na CAR rows: (72, 19)


In [66]:
import statsmodels.formula.api as smf

model = smf.ols(
    formula="""
        CAR_m1_p1 ~ 
        franchise_gta +
        is_rockstar +
        market_return +
        AR_event +
        C(event_type) +
        C(publisher) +
        C(sentiment)
    """,
    data=final_df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              CAR_m1_p1   R-squared:                       0.484
Model:                            OLS   Adj. R-squared:                  0.294
Method:                 Least Squares   F-statistic:                     2.551
Date:                Mon, 24 Nov 2025   Prob (F-statistic):            0.00487
Time:                        12:08:54   Log-Likelihood:                 145.43
No. Observations:                  68   AIC:                            -252.9
Df Residuals:                      49   BIC:                            -210.7
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [67]:
# Make sure we work on the cleaned df (the same one used for OLS)
gta6_mask = df["game"].astype(str).str.lower().str.contains("gta vi", na=False)

gta6_events = df.loc[gta6_mask].copy()

gta6_events[["event_id", "event_date", "game", "event_type", "sentiment", "CAR_m1_p1"]]

,event_id,event_date,game,event_type,sentiment,CAR_m1_p1
49,TTWO_2022_GTA6_DEV-ANNOUNCEMENT,2022-02-04,GTA VI,major announcement,neutral,0.078164
52,GTA6_2022_LEAK,2022-09-18,GTA VI,leak,negative,-0.014621
55,TTWO_2023_GTA6_TRAILER1,2023-12-05,GTA VI,trailer/reveal,positive,0.004367
58,GTA6_2024_KEYART,2024-05-16,GTA VI,trailer/reveal,positive,0.008283
59,TTWO_2025_GTA6_DELAY1,2025-05-02,GTA VI,delay,negative,-0.079546
60,TTWO_2025_GTA6_TRAILER2,2025-05-06,GTA VI,trailer/reveal,positive,0.039229
61,TTWO_2025_GTA6_DELAY2,2025-11-06,GTA VI,delay,negative,-0.076612


In [69]:
# Use the same model you fitted above
gta6_events["CAR_hat"] = model.predict(gta6_events)

# Prediction error
gta6_events["error"] = gta6_events["CAR_m1_p1"] - gta6_events["CAR_hat"]

# Did the model get the sign right?
gta6_events["sign_match"] = (
    gta6_events["CAR_m1_p1"] * gta6_events["CAR_hat"] > 0
)

# Round for readability
cols_to_show = [
    "event_id",
    "event_date",
    "game",
    "event_type",
    "sentiment",
    "CAR_m1_p1",
    "CAR_hat",
    "error",
    "sign_match",
]

gta6_events[cols_to_show].round(4)

,event_id,event_date,game,event_type,sentiment,CAR_m1_p1,CAR_hat,error,sign_match
49,TTWO_2022_GTA6_DEV-ANNOUNCEMENT,2022-02-04,GTA VI,major announcement,neutral,0.0782,0.0498,0.0283,True
52,GTA6_2022_LEAK,2022-09-18,GTA VI,leak,negative,-0.0146,-0.0146,-0.0000,True
55,TTWO_2023_GTA6_TRAILER1,2023-12-05,GTA VI,trailer/reveal,positive,0.0044,-0.0072,0.0115,False
58,GTA6_2024_KEYART,2024-05-16,GTA VI,trailer/reveal,positive,0.0083,-0.0145,0.0228,False
59,TTWO_2025_GTA6_DELAY1,2025-05-02,GTA VI,delay,negative,-0.0795,-0.0816,0.0020,True
60,TTWO_2025_GTA6_TRAILER2,2025-05-06,GTA VI,trailer/reveal,positive,0.0392,0.0314,0.0079,True
61,TTWO_2025_GTA6_DELAY2,2025-11-06,GTA VI,delay,negative,-0.0766,-0.0072,-0.0694,True


In [70]:
import pandas as pd

# Build future GTA VI release scenarios
scenarios = pd.DataFrame([
    {
        "scenario": "bear",
        "AR_event": -0.03,   # -3% day-zero shock
        "market_return": 0.0,
        "franchise_gta": 1,
        "is_rockstar": 1,
        "event_type": "release",
        "sentiment": "negative",
        "publisher": "take-two",
    },
    {
        "scenario": "base",
        "AR_event": 0.00,    # neutral opening day
        "market_return": 0.0,
        "franchise_gta": 1,
        "is_rockstar": 1,
        "event_type": "release",
        "sentiment": "neutral",
        "publisher": "take-two",
    },
    {
        "scenario": "bull",
        "AR_event": +0.03,   # +3% day-zero hype
        "market_return": 0.0,
        "franchise_gta": 1,
        "is_rockstar": 1,
        "event_type": "release",
        "sentiment": "positive",
        "publisher": "take-two",
    }
])

# Use the same encoding as the regression
scenarios_pred = model.predict(scenarios)

pd.DataFrame({
    "scenario": scenarios["scenario"],
    "Predicted CAR_m1_p1": scenarios_pred
}).round(4)

,scenario,Predicted CAR_m1_p1
0,bear,-0.0544
1,base,-0.0267
2,bull,0.0127
